In [1]:
!pip install --upgrade kaggle -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 8.7 MB/s eta 0:00:00


In [ ]:
!kaggle kernels output pankajmaulekhi/training-model -p /kaggle/working/

Output file downloaded to /kaggle/working/checkpoint.pth
Output file downloaded to /kaggle/working/loss_vs_steps.png
Output file downloaded to /kaggle/working/loss_vs_time.png
Output file downloaded to /kaggle/working/my_tokenizer.json


In [13]:
import torch
import matplotlib.pyplot as plt
import plotly.express as px
from tokenizers import Tokenizer
from train import LLM, vocab_size, embed_dim, n_layer, n_head

device = "cuda" if torch.cuda.is_available() else "cpu"

In [14]:
checkpoint = torch.load(
    "/kaggle/working/checkpoint.pth", map_location="cpu", weights_only=False
)

In [15]:
checkpoint.keys()

dict_keys(['epoch', 'model_state_dict', 'optimiser_state_dict', 'loss_scaler', 'lr_scheduler_state_dict', 'loss_per_step_lst', 'time_per_step', 'rng_state', 'numpy_rng_state'])

In [16]:
loss = checkpoint["loss_per_step_lst"]

In [ ]:
fig = px.line(loss, log_y=True)
fig.show()

In [17]:
tokenizer = Tokenizer.from_file("/kaggle/working/my_tokenizer.json")
model = LLM(vocab_size, embed_dim, n_layer, n_head)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

LLM(
  (embedding): Embedding(32000, 768)
  (embed_dropout): Dropout(p=0.1, inplace=False)
  (llm_layers): ModuleList(
    (0-11): 12 x Llm_layer(
      (linear1): Linear(in_features=768, out_features=2304, bias=True)
      (attn_layer): ScaledDotProductAttention(
        (softmax): Softmax(dim=-1)
      )
      (lyr_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (linear2): Linear(in_features=768, out_features=768, bias=True)
      (lyr_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=768, out_features=3072, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=3072, out_features=768, bias=True)
      )
      (rope): RotaryPositionalEmbeddings()
      (linear_dropout): Dropout(p=0.1, inplace=False)
      (ffn_dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)

In [18]:
tokenizer.encode("What is asset?").ids

[22266, 1236, 1504, 36]

In [19]:
tokenizer.get_added_tokens_decoder()

{0: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 1: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 2: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 3: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 4: AddedToken("[EOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 5: AddedToken("[BOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True)}

In [14]:
eos_token_id = tokenizer.encode("[EOS]").ids[0]


def generate(model, tokenizer, inp, eos_token_id, max_new_token=100):
    inp = torch.tensor([tokenizer.encode(inp).ids]).to(device)
    i = 0
    while inp[0][-1].item() != eos_token_id:
        i += 1
        with torch.no_grad():
            res = model(inp)
        new_id = (
            torch.nn.functional.softmax(res[0][-1], dim=-1)
            .argmax(dim=-1)
            .unsqueeze(0)
            .unsqueeze(0)
        )
        print(tokenizer.decode(new_id.tolist()[0]), end=" ")
        if i == max_new_token:
            print("\n MAX TOKEN REACHED")
            break
        inp = torch.cat([inp, new_id], dim=-1)
    # return tokenizer.decode(inp.tolist()[0])

In [15]:
generate(
    model,
    tokenizer,
    "The company reported a net revenue of $4.2 billion, representing a year-over-year growth of   ",
    eos_token_id,
)

$ 1 . 1 billion , or 6 . 1 %, compared to the prior year . The increase in revenue was primarily due to the inclusion of the company ’ s results of operations for the year ended December 31 , 2016 , as compared to the prior year . The company reported a net revenue of $ 3 . 1 billion , representing a year - over - year growth of $ 1 . 1 billion , or 6 . 1 %, compared to the prior year . The increase in revenue was primarily due to the inclusion 
 MAX TOKEN REACHED


In [16]:
inputs = [
    "The company reported total revenue of $",
    "Operating margin improved by",
    "Earnings per share for fiscal year 2023 was",
    "The board declared a dividend of",
    "Capital expenditure for the year amounted to",
    "The company is exposed to significant risks including",
    "Fluctuations in foreign exchange rates may adversely",
    "The following factors could materially affect our",
]

for inp in inputs:
    print(f"INPUT -", inp)
    print("Output -", end=" ")
    generate(model, tokenizer, inp, eos_token_id)
    print()
    print()

INPUT - The company reported total revenue of $
Output - 1 , 000 , 000 for the year ended December 31 , 2009 , compared to $ 1 , 000 , 000 for the year ended December 31 , 2008 . The decrease in revenue was primarily due to a decrease in the number of customers and a decrease in the number of customers . The company ’ s revenue for the year ended December 31 , 2009 was $ 1 , 000 , 000 compared to $ 1 , 000 , 000 for the year ended December 31 , 2008 . The decrease in revenue was primarily due to 
 MAX TOKEN REACHED


INPUT - Operating margin improved by
Output - $ 1 . 6 million , or 1 . 6 %, to $ 16 . 6 million for the year ended December 31 , 2015 from $ 16 . 6 million for the year ended December 31 , 2014 . The increase in operating margin was primarily due to the increase in operating expenses , partially offset by the decrease in interest expense and the decrease in interest income . The increase in operating expenses was primarily due to the increase in operating expenses , partia

KeyboardInterrupt: 

# Pushing to Hugging face

In [20]:
from transformers import PretrainedConfig
from transformers import PreTrainedModel, GenerationMixin

In [21]:
!mkdir /kaggle/working/PM_MiniFinLLM

mkdir: cannot create directory ‘/kaggle/working/PM_MiniFinLLM’: File exists


In [22]:
vocab_size, embed_dim, n_layer, n_head

(32000, 768, 12, 12)

In [23]:
%%writefile /kaggle/working/PM_MiniFinLLM/arch.py

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from tokenizers import Tokenizer
import numpy as np
from torch.utils.data import Dataset,DataLoader
from torch.cuda.amp import autocast,GradScaler
import math
import time
from torch.optim.lr_scheduler import LambdaLR
import matplotlib.pyplot as plt
from torch.utils.checkpoint import checkpoint
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group,destroy_process_group
import contextlib
from IPython.display import display, Image
import torch.multiprocessing as mp

"""# DDP setup"""

def ddp_setup(rank,world_size):
    os.environ["MASTER_ADDR"] = 'localhost'
    os.environ["MASTER_PORT"] = '12355'
    init_process_group(backend='nccl',rank=rank,world_size=world_size)

"""# Global Vars"""

dtype=torch.float16
vocab_size= 32000
embed_dim= 768
n_head= 12
n_layer= 12
device='cuda' if torch.cuda.is_available() else 'cpu'
batch_size= 12
epochs = 1

max_seq_len=1024

weight_decay=0.1
b1=0.9
b2=0.95
peak_lr=3e-4
gradient_accumulation_steps= 4
BASE_DIR='/kaggle/working/'
update_checkpoint_steps=100
update_plot_steps=100

"""# RoPE"""

class RotaryPositionalEmbeddings(nn.Module):

  def __init__(self, d: int, base: int = 10_000):

    super().__init__()
    self.base = base
    self.d = d
    self.cos_cached = None
    self.sin_cached = None
  def _build_cache(self, x: torch.Tensor):
      seq_len = x.shape[1]  # B, T, heads, d -> T is dim 1

      if self.cos_cached is not None and seq_len <= self.cos_cached.shape[0]:
          return

      theta = 1. / (self.base ** (torch.arange(0, self.d, 2).float() / self.d)).to(x.device)
      seq_idx = torch.arange(seq_len, device=x.device).float()
      idx_theta = torch.einsum('n,d->nd', seq_idx, theta)
      idx_theta2 = torch.cat([idx_theta, idx_theta], dim=1)

      self.cos_cached = idx_theta2.cos()[None, :, None, :]  # (1, T, 1, d)
      self.sin_cached = idx_theta2.sin()[None, :, None, :]  # (1, T, 1, d)

  def _neg_half(self, x: torch.Tensor):
      d_2 = self.d // 2
      return torch.cat([-x[:, :, :, d_2:], x[:, :, :, :d_2]], dim=-1)

  def forward(self, x: torch.Tensor):
      # x: (B, T, num_heads, head_dim)
      self._build_cache(x)
      neg_half_x = self._neg_half(x)
      x_rope = (x * self.cos_cached[:, :x.shape[1]]) + (neg_half_x * self.sin_cached[:, :x.shape[1]])
      return x_rope

"""# Attention"""

class ScaledDotProductAttention(nn.Module):
  def __init__(self,per_head_embed_dim):
    super().__init__()
    self.softmax=nn.Softmax(dim=-1)
    self.d=per_head_embed_dim

  def forward(self,Q,K,V):
    return nn.functional.scaled_dot_product_attention(Q,K,V,is_causal=True)

"""# Layer"""

class Llm_layer(nn.Module):
  def __init__(self,n_head,embed_dim,dropout_ratio=0.1):
    super().__init__()
    self.embed_dim=embed_dim
    self.n_head=n_head
    self.linear1=nn.Linear(embed_dim,embed_dim*3)
    self.attn_layer=ScaledDotProductAttention(per_head_embed_dim=embed_dim//n_head)
    self.lyr_norm1=nn.LayerNorm(embed_dim)
    self.linear2=nn.Linear(embed_dim,embed_dim)
    self.lyr_norm2=nn.LayerNorm(embed_dim)
    self.ffn=nn.Sequential(nn.Linear(embed_dim,embed_dim*4),nn.GELU(),nn.Dropout(p=dropout_ratio),nn.Linear(embed_dim*4,embed_dim))
    self.rope=RotaryPositionalEmbeddings(d=embed_dim//n_head)
    self.linear_dropout=nn.Dropout(p=dropout_ratio)
    self.ffn_dropout=nn.Dropout(p=dropout_ratio)


  def forward(self,x):
    x0=self.linear1(self.lyr_norm1(x))
    Q,K,V=x0.split(self.embed_dim,dim=-1)

    Q=self.rope(Q.reshape(Q.shape[0],Q.shape[1],self.n_head,self.embed_dim//self.n_head)).transpose(1,2)
    K=self.rope(K.reshape(K.shape[0],K.shape[1],self.n_head,self.embed_dim//self.n_head)).transpose(1,2)
    V=V.reshape(V.shape[0],V.shape[1],self.n_head,self.embed_dim//self.n_head).transpose(1,2)

    res=self.attn_layer(Q,K,V)
    a,b,c,d=res.shape
    res=res.transpose(1,2).reshape(a,c,self.embed_dim)

    x=x+self.linear_dropout(self.linear2(res))

    x=x+self.ffn_dropout(self.ffn(self.lyr_norm2(x)))
    return x

"""# LLM"""

# @title LLM
class LLM(nn.Module):
  def __init__(self,vocab_size,embed_dim,n_layer,n_head,dropout_ratio=0.1):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embed_dim)
    self.embed_dropout = nn.Dropout(p=dropout_ratio)
    self.llm_layers=nn.ModuleList([Llm_layer(n_head=n_head,embed_dim=embed_dim) for i in range(n_layer)])
    self.layer_norm1=nn.LayerNorm(embed_dim)
    self.linear=nn.Linear(embed_dim,vocab_size)
    self.linear.weight=self.embedding.weight
    self.softmax=nn.Softmax(dim=-1)

  def forward(self,x):
    x=self.embed_dropout(self.embedding(x))
    for llm_layer in self.llm_layers:
      x=llm_layer(x)
    x=self.linear(self.layer_norm1(x))

    return x

Writing /kaggle/working/PM_MiniFinLLM/arch.py


In [24]:
%%writefile /kaggle/working/PM_MiniFinLLM/configuration_PM_MiniFinLLM.py

from transformers import PretrainedConfig

class PM_MiniFinLLM_config(PretrainedConfig):
    model_type = "PM_MiniFinLLM"
    def __init__(self,vocab_size=32000,embed_dim=768,n_layer=12,n_head=12,**kwargs):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.n_layer = n_layer
        self.n_head = n_head
        super().__init__(**kwargs) 

Writing /kaggle/working/PM_MiniFinLLM/configuration_PM_MiniFinLLM.py


In [25]:
!cp /kaggle/working/train.py /kaggle/working/PM_MiniFinLLM

In [70]:
%%writefile /kaggle/working/PM_MiniFinLLM/modeling_PM_MiniFinLLM.py

from transformers import PreTrainedModel,GenerationMixin
from .configuration_PM_MiniFinLLM import PM_MiniFinLLM_config
from transformers.modeling_outputs import CausalLMOutputWithPast
from .arch import LLM,vocab_size,embed_dim,n_layer,n_head
import torch.nn as nn
import torch


class PM_MiniFinLLM_Model(PreTrainedModel,GenerationMixin):
    config_class=PM_MiniFinLLM_config
    _tied_weights_keys = {
        "model.linear.weight": "model.embedding.weight"  #  duplicate -> source
    }
    def __init__(self,config):
        super().__init__(config)
        self.model=LLM(config.vocab_size,config.embed_dim,config.n_layer,config.n_head)
        self.post_init()
        self.cross_entropy_loss=nn.CrossEntropyLoss(ignore_index=-100)
        
    def loss_fn(self,y_pred,y_true):
      return self.cross_entropy_loss(y_pred.reshape(-1,32000),y_true.reshape(-1))

    def tie_weights(self,*args,**kwargs):
        self.model.linear.weight = self.model.embedding.weight

    def prepare_inputs_for_generation(self, input_ids, **kwargs):
        return {"input_ids": input_ids}

    def forward(self, input_ids, labels=None, **kwargs):
        logits=self.model(input_ids)
        loss=None

        if labels is not None:
            shifted_logits=logits[:,:-1,:]
            shifted_labels=labels[:,1:]
            loss=self.loss_fn(shifted_logits,shifted_labels)

        return CausalLMOutputWithPast(loss=loss,logits=logits,past_key_values=None)

Overwriting /kaggle/working/PM_MiniFinLLM/modeling_PM_MiniFinLLM.py


In [71]:
!touch /kaggle/working/PM_MiniFinLLM/__init__.py

In [72]:
from PM_MiniFinLLM.modeling_PM_MiniFinLLM import PM_MiniFinLLM_Model
from PM_MiniFinLLM.configuration_PM_MiniFinLLM import PM_MiniFinLLM_config

PM_MiniFinLLM_config.register_for_auto_class()
PM_MiniFinLLM_Model.register_for_auto_class("AutoModelForCausalLM")

In [73]:
config = PM_MiniFinLLM_config(vocab_size, embed_dim, n_layer, n_head)
model = PM_MiniFinLLM_Model(config)
model.model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [74]:
print(list(checkpoint["model_state_dict"].keys())[:5])
print(list(model.state_dict().keys())[:5])

['embedding.weight', 'llm_layers.0.linear1.weight', 'llm_layers.0.linear1.bias', 'llm_layers.0.lyr_norm1.weight', 'llm_layers.0.lyr_norm1.bias']
['model.embedding.weight', 'model.llm_layers.0.linear1.weight', 'model.llm_layers.0.linear1.bias', 'model.llm_layers.0.lyr_norm1.weight', 'model.llm_layers.0.lyr_norm1.bias']


In [75]:
print(model.state_dict()["model.embedding.weight"][0][:5])

tensor([-0.1794, -0.4016, -0.3976, -0.5725,  0.2631])


In [76]:
print(model.model.embedding.weight == model.model.linear.weight)

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])


In [ ]:
!hf auth login --token your_hf token

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `MyLLm` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `MyLLm`


In [78]:
model.can_generate()

True

In [53]:
from transformers import TextStreamer

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

In [59]:
model.generate(
    torch.tensor(tokenizer.encode("What is asset?").ids).unsqueeze(0),
    streamer=streamer,
)

In [51]:
from transformers import PreTrainedTokenizerFast

tokenizer_hf = PreTrainedTokenizerFast(tokenizer_object=tokenizer)

special_tokens = ["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[EOS]", "[BOS]"]

tokenizer_hf = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    # because i forgot to add <user>,</user> so i am using these special tokens as them
    user_start_token="[BOS]",
    user_end_token="[CLS]",
    assistant_token="[SEP]",
    eos_token="[EOS]",
)

In [65]:
tokenizer_hf.chat_template = """{% for msg in messages %}{% if msg['role'] == 'user' %}[BOS] {{ msg['content'] }} [CLS]{% elif msg['role'] == 'assistant' %}{% generation %}[SEP] {{ msg['content'] }} [EOS]{% endgeneration %}{% endif %}{% endfor %}""".strip()

In [66]:
print(tokenizer_hf.special_tokens_map)

{'eos_token': '[EOS]', 'unk_token': '[UNK]', 'pad_token': '[PAD]', 'user_start_token': '[BOS]', 'user_end_token': '[CLS]', 'assistant_token': '[SEP]'}


In [67]:
x = [
    {
        "content": "Should I cash out my Roth IRA to pay my mother's property tax debt, to avoid foreclosure on her home?",
        "role": "user",
    },
    {
        "content": "You're crazy to cash out your Roth or take on 401k loan, as that is addressing a short-term problem without doing anything about the longer-term issue. Just don't do it. Through no fault of her own, your mom is insolvent. It happens to people all of the time, and the solution is chapter 7 bankruptcy.  The only thing that I would do with my money in this situation is help her with bankruptcy attorney fees if needed, and maybe bid on it at auction, if the house in in good shape.",
        "role": "assistant",
    },
]
out = tokenizer_hf.apply_chat_template(
    x,
    return_dict=True,
    return_assistant_tokens_mask=True,  # <-- this is the key
    tokenize=True,
)

print(tokenizer_hf.decode(out["input_ids"]))
print(out["assistant_masks"])  # 1 = assistant token, 0 = masked out

[BOS] Should I cash out my Roth IRA to pay my mother ' s property tax debt , to avoid foreclosure on her home ? [CLS] [SEP] You ' re cra zy to cash out your Roth or take on 401 k loan , as that is addressing a short - term problem without doing anything about the longer - term issue . Just don ' t do it . Through no fault of her own , your mo m is insolvent . It happens to people all of the time , and the solution is chapter 7 bankruptcy . The only thing that I would do with my money in this situation is help her with bankruptcy attorney fees if needed , and may be bid on it at auction , if the house in in good shape . [EOS]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [68]:
print(tokenizer_hf.chat_template)

{% for msg in messages %}{% if msg['role'] == 'user' %}[BOS] {{ msg['content'] }} [CLS]{% elif msg['role'] == 'assistant' %}{% generation %}[SEP] {{ msg['content'] }} [EOS]{% endgeneration %}{% endif %}{% endfor %}


In [69]:
tokenizer_hf.push_to_hub("Pankaj121212/PM_MiniFinLLM")

CommitInfo(commit_url='https://huggingface.co/Pankaj121212/PM_MiniFinLLM/commit/8cc52061bb164d7c1dc1f761a7ca01bb9f496770', commit_message='Upload tokenizer', commit_description='', oid='8cc52061bb164d7c1dc1f761a7ca01bb9f496770', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Pankaj121212/PM_MiniFinLLM', endpoint='https://huggingface.co', repo_type='model', repo_id='Pankaj121212/PM_MiniFinLLM'), pr_revision=None, pr_num=None)

In [79]:
model.push_to_hub("Pankaj121212/PM_MiniFinLLM")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Pankaj121212/PM_MiniFinLLM/commit/ff8b7c0efc5d1302c7dcc492762082ca023909ed', commit_message='Upload model', commit_description='', oid='ff8b7c0efc5d1302c7dcc492762082ca023909ed', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Pankaj121212/PM_MiniFinLLM', endpoint='https://huggingface.co', repo_type='model', repo_id='Pankaj121212/PM_MiniFinLLM'), pr_revision=None, pr_num=None)

In [35]:
# uploading train.py
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="/kaggle/working/PM_MiniFinLLM/arch.py",  # local path
    path_in_repo="arch.py",  # where it lands in the repo
    repo_id="Pankaj121212/PM_MiniFinLLM",
    repo_type="model",
)

CommitInfo(commit_url='https://huggingface.co/Pankaj121212/PM_MiniFinLLM/commit/5147b6d73df74db7320be80b61c1bb946b264d33', commit_message='Upload train.py with huggingface_hub', commit_description='', oid='5147b6d73df74db7320be80b61c1bb946b264d33', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Pankaj121212/PM_MiniFinLLM', endpoint='https://huggingface.co', repo_type='model', repo_id='Pankaj121212/PM_MiniFinLLM'), pr_revision=None, pr_num=None)